In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from services.google_services_own_account import (
    connect_google,
    read_excel,
    read_sheet,
    write_sheet,
)

drive, sheets = connect_google()

In [2]:
FILE_EXCEL_ID = "1rxJcxH482wk-Z7H3oW76LqzrfVS_H1hf"

file_buffer_excel = read_excel(
    FILE_EXCEL_ID,
    drive
)


Berhasil


In [3]:
import pandas as pd, csv, io
from openpyxl import load_workbook

path = file_buffer_excel
ws = load_workbook(path, read_only=True).active

text = "\n".join(str(r[0]) for r in ws.iter_rows(values_only=True) if r[0] is not None)
rows = list(csv.reader(io.StringIO(text)))

header, data = rows[0], rows[1:]
good = [r for r in data if len(r) == len(header)]
bad  = [r for r in data if len(r) != len(header)]

df = pd.DataFrame(good, columns=header).replace("", pd.NA)
print("Valid:", len(df), "| Bermasalah:", len(bad))
df.head()

Valid: 46598 | Bermasalah: 269


,ktun_id,no_ktun,no_reg,kbli,fas_id,instansi,nib,master_sumber_id,no_reg_sumber,status_ktun,...,status_sumber,ket_status,nama_proyek,alamat,alamat_id,kab_id,kab_kota,propinsi,posisi_sumber,versi_balis
0,32314,06299.473.1.260624,207183.24,NaN,13349,Badan Riset dan Inovasi Nasional,NaN,77699,18.2.0077699,Berakhir,...,Aktif,t.dewi- Balis Inspeksi,BRIN kawasan Yogyakarta,Jalan Babarsari PO Box 6101 YKBB,40467,294,Sleman,Daerah Istimewa Yogyakarta,NaN,balis2.5
1,32314,06299.473.1.260624,207183.24,NaN,13349,Badan Riset dan Inovasi Nasional,NaN,77704,18.2.0077704,Berakhir,...,Aktif,t.dewi- Balis Inspeksi,BRIN kawasan Yogyakarta,Jalan Babarsari PO Box 6101 YKBB,40467,294,Sleman,Daerah Istimewa Yogyakarta,NaN,balis2.5
2,32314,06299.473.1.260624,207183.24,NaN,13349,Badan Riset dan Inovasi Nasional,NaN,77749,18.2.0077749,Berakhir,...,Aktif,t.dewi- Balis Inspeksi,BRIN kawasan Yogyakarta,Jalan Babarsari PO Box 6101 YKBB,40467,294,Sleman,Daerah Istimewa Yogyakarta,NaN,balis2.5
3,32314,06299.473.1.260624,207183.24,NaN,13349,Badan Riset dan Inovasi Nasional,NaN,77750,18.2.0077750,Berakhir,...,Aktif,t.dewi- Balis Inspeksi,BRIN kawasan Yogyakarta,Jalan Babarsari PO Box 6101 YKBB,40467,294,Sleman,Daerah Istimewa Yogyakarta,NaN,balis2.5
4,32314,06299.473.1.260624,207183.24,NaN,13349,Badan Riset dan Inovasi Nasional,NaN,77764,18.2.0077764,Berakhir,...,Aktif,t.dewi- Balis Inspeksi,BRIN kawasan Yogyakarta,Jalan Babarsari PO Box 6101 YKBB,40467,294,Sleman,Daerah Istimewa Yogyakarta,NaN,balis2.5


In [4]:
df_valid = df.copy()

df_valid = df_valid.drop_duplicates()
df_valid = df_valid[
    df_valid["no_seri"].notna()
    & df_valid["no_seri"].astype(str).str.strip().ne("")
]
df_valid = df_valid[
    df_valid["propinsi"].notna()
    & df_valid["propinsi"].astype(str).str.strip().ne("")
]

In [5]:
df_valid.shape[0]

46254

In [6]:
df_total = (
    df_valid
    .groupby(["propinsi", "jenis_sumber"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["PRP", "ZRA"], fill_value=0)
)

df_total["Total"] = df_total["PRP"] + df_total["ZRA"]

df_total = (
    df_total
    .rename(columns={
        "PRP": "Jumlah PRP",
        "ZRA": "Jumlah ZRA",
        "Total": "Total SRP"
    })
    .reset_index()
    .rename(columns={"propinsi": "Provinsi"})
)

In [13]:
FILE_REKAPITULASI_ID = "1jZt22z352D151rMTqZITiPbenX45VIC7iTWe79VyjsA"

write_sheet(
    df_total,
    FILE_REKAPITULASI_ID,
    range_name="Sheet1!B1",
    sheets=sheets
)